# 08c — Análise secundária: avaliação e univariada × multivariada

**Lê:** `previsoes_teste` (05), `secundaria_pred_*` (08a, 08b), `selecao_modelo.json`, histórico do treino e a série completa (trava, *secundaria*).  
**Grava:** tabelas `secundaria_*` e figuras.

Compare cada par **dentro do mesmo nível**. Só afirme que as exógenas ajudam quando o IC95 de `d` ficar todo abaixo de zero.

Inclui o **seasonal naive** (`SNaive_52`, mesma semana do ano anterior) como referência complementar: não entra na seleção porque as origens de validação de 2017 não têm 52 semanas de histórico; onde falta t−52, usa o Naive.

In [ ]:
# === SETUP (idêntico em todos os notebooks; só muda NOTEBOOK) ===
NOTEBOOK = "08c_secundaria_avaliacao"
RAIZ = "/content/drive/MyDrive/tcc_fluxo"      # Colab. Localmente: caminho da pasta do projeto (ou deixe None)

import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip install -q -r {RAIZ}/requirements.txt
except ImportError:                              # fora do Colab
    RAIZ = os.environ.get("TCC_RAIZ") or None
    if RAIZ is None:
        from pathlib import Path
        RAIZ = str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").is_file()))
sys.path.insert(0, f"{RAIZ}/src")
%load_ext autoreload
%autoreload 2

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tcc_fluxo import iniciar
C, A = iniciar(RAIZ, notebook=NOTEBOOK)

In [ ]:
from tcc_fluxo import secundaria as S, metricas as M, graficos

sel = A.carregar_json("selecao_modelo")
CHOSEN = sel["modelo_escolhido"]
comparison = A.carregar_df("previsoes_teste")
hist = A.carregar_historico_treino()
ts_pred = A.carregar_df("secundaria_pred_vendedor_estat")
lstm = A.carregar_df("secundaria_pred_lstm")
agg_pred = A.carregar_df("secundaria_pred_agregado")
scale_final = M.seller_scales(hist, C.TRAIN_TARGET_END)
agg_scale = M.aggregate_scale(hist, C.TRAIN_TARGET_END)
_, aggregate = M.evaluate_aggregate(comparison, ["Naive", CHOSEN], agg_scale)

# Seasonal naive (mesma semana do ano anterior): referência complementar, só no teste.
serie = A.carregar_serie_completa(finalidade="secundaria")
pred_snaive, disp_snaive = S.seasonal_naive(comparison, serie)
snaive_tab, snaive_cob = S.avaliar_snaive(comparison, CHOSEN, pred_snaive, disp_snaive, scale_final, agg_scale)
A.tabela(snaive_tab, "secundaria_seasonal_naive.csv")
A.salvar_json(snaive_cob, "secundaria_seasonal_naive_cobertura")
print("Seasonal naive — cobertura:", snaive_cob); display(snaive_tab)

res = S.avaliar(comparison, CHOSEN, ts_pred, lstm, agg_pred, aggregate, scale_final, agg_scale,
                extras={"SNaive_52": pred_snaive})
A.tabela(res["vendedor"], "secundaria_metricas_vendedor.csv"); A.tabela(res["agregado"], "secundaria_metricas_agregado.csv")
A.tabela(res["per_seller"], "secundaria_metricas_por_vendedor.csv")
print("Cortes em zero dos modelos lineares:"); display(res["clip"])
display(res["vendedor"]); display(res["agregado"])

In [ ]:
um = S.univariada_vs_multivariada(res, C.N_BOOT, C.RANDOM_SEED, C.MIN_WEEKS_DM)
A.tabela(um, "secundaria_univariada_vs_multivariada.csv")
display(um[["nivel", "familia", "univariada", "multivariada", "n", "univ", "multi", "d_media", "ic95_inf", "ic95_sup",
            "prop_vendedores_multi_melhor", "p_valor_hln", "leitura"]])

In [ ]:
C_UNI, C_MULTI, C_BASE = "#2a78d6", "#1baf7a", "#b4b2a9"
col = lambda t: {"univariada": C_UNI, "multivariada": C_MULTI}.get(t, C_BASE)
fig, axes = plt.subplots(1, 2, figsize=(13, 5.4))
d_s = res["vendedor"].sort_values("MASE_media", ascending=False)
axes[0].barh(d_s["modelo"], d_s["MASE_media"], color=[col(t) for t in d_s["tipo"]])
axes[0].axvline(float(res["vendedor"].set_index("modelo").loc["Naive", "MASE_media"]), color=graficos.INK2, ls="--", lw=1)
axes[0].set_title("Nível vendedor — MASE médio por vendedor (teste)", loc="left")
d_a = res["agregado"].sort_values("MASE", ascending=False)
axes[1].barh(d_a["modelo"], d_a["MASE"], color=[col(t) for t in d_a["tipo"]])
axes[1].axvline(float(res["agregado"].set_index("modelo").loc["Naive (soma por vendedor)", "MASE"]), color=graficos.INK2, ls="--", lw=1)
axes[1].set_title("Nível agregado — MASE do total semanal (teste)", loc="left")
for ax in axes:
    graficos.clean(ax); ax.grid(axis="y", visible=False); ax.tick_params(axis="y", labelsize=8.5)
fig.tight_layout(); A.figura(fig, "secundaria_comparacao.png", bbox_inches="tight"); plt.show()

In [ ]:
A.finalizar()